<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW14 · Session 2 · Cross-Validation, Additional Classifiers, and Regression

        **Course level:** developing beginner  
        **Prior learning:** classification pipelines and holdout metrics from TW14 Session 1  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        A single holdout result can vary by chance. We compare LDA, Gaussian Naive Bayes, and SVC with cross-validation, then complete a regression walkthrough with MAE, RMSE, and R-squared.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - explain how cross-validation differs from a final test set
- train LDA, Gaussian Naive Bayes, and SVC pipelines
- compare models using stratified cross-validation
- distinguish classification from regression
- evaluate regression with MAE, RMSE, and R-squared


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Load classification tools and data

Dense one-hot output supports all three classifiers in this session.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

churn_data = pd.read_csv(DATA_DIR / "customer_churn.csv")
X = churn_data.drop(columns=["customer_id", "churn"])
y = churn_data["churn"]
print(X.shape, y.mean())


## 1. Cross-validation

k-fold cross-validation divides training data into k parts. Each part becomes validation data once while the others train the model. It estimates variability more reliably than one validation split.

The final test set should remain untouched until model choices are complete.


### Create preprocessing and model candidates

The preprocessing remains inside every pipeline so each fold learns transformations from its training portion only.


In [ ]:
numeric_columns = ["age", "tenure_months", "monthly_spend", "support_tickets"]
categorical_columns = ["contract_type", "autopay"]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_columns),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_columns),
])

classification_models = {
    "LDA": LinearDiscriminantAnalysis(),
    "Gaussian NB": GaussianNB(),
    "SVC": SVC(kernel="rbf", C=1.0, gamma="scale"),
}
print(classification_models)


## 2. Three additional classifiers

- LDA models class separation using linear combinations under distributional assumptions.
- Gaussian Naive Bayes assumes conditionally independent features with Gaussian numerical distributions.
- SVC finds a separating boundary; an RBF kernel can model non-linear structure.

Their assumptions differ, so evaluation matters more than popularity.


### Run stratified five-fold evaluation

We compare accuracy, recall, and F1 and include fit-time information.


In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
classification_rows = []

for name, estimator in classification_models.items():
    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", estimator),
    ])
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=stratified_cv,
        scoring=["accuracy", "precision", "recall", "f1"],
    )
    classification_rows.append({
        "model": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_sd": scores["test_accuracy"].std(),
        "precision_mean": scores["test_precision"].mean(),
        "recall_mean": scores["test_recall"].mean(),
        "f1_mean": scores["test_f1"].mean(),
    })

cv_comparison = pd.DataFrame(classification_rows).set_index("model")
print(cv_comparison.round(3))


### Inspect fold-level variability

Mean performance without variation can hide unstable results.


In [ ]:
svc_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", SVC(kernel="rbf")),
])
svc_scores = cross_validate(
    svc_pipeline,
    X,
    y,
    cv=stratified_cv,
    scoring="f1",
    return_train_score=True,
)
print("Train F1:", np.round(svc_scores["train_score"], 3))
print("Validation F1:", np.round(svc_scores["test_score"], 3))


## 3. Classification versus regression

Classification predicts a category such as churn or retained. Regression predicts a continuous number such as weekly revenue. Metrics are therefore different.


### Generate a reproducible business regression dataset

Synthetic store features create a transparent teaching relationship plus noise.


In [ ]:
rng = np.random.default_rng(42)
n_stores = 240
regression_data = pd.DataFrame({
    "floor_area": rng.normal(180, 45, n_stores).clip(70, 320),
    "staff_hours": rng.normal(220, 55, n_stores).clip(80, 400),
    "marketing_budget": rng.gamma(3, 700, n_stores),
    "weekly_footfall": rng.normal(950, 220, n_stores).clip(300, 1700),
})
regression_data["weekly_revenue"] = (
    35 * regression_data["floor_area"]
    + 18 * regression_data["staff_hours"]
    + 1.4 * regression_data["marketing_budget"]
    + 22 * regression_data["weekly_footfall"]
    + rng.normal(0, 2500, n_stores)
)
print(regression_data.head())


### Create a regression holdout split

Stratification is not used for a continuous target in this simple walkthrough.


In [ ]:
X_reg = regression_data.drop(columns="weekly_revenue")
y_reg = regression_data["weekly_revenue"]
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.25,
    random_state=42,
)
print(X_reg_train.shape, X_reg_test.shape)


## 4. Train and evaluate regression models

- MAE is the mean absolute error in the target unit.
- RMSE is the square root of mean squared error and penalises larger errors more strongly.
- R-squared compares explained variation with predicting the target mean; it can be negative on test data.


### Fit linear and tree regression

Scaling is included for the linear model. The tree is constrained to limit complexity.


In [ ]:
linear_regression = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
tree_regression = DecisionTreeRegressor(
    max_depth=5,
    min_samples_leaf=8,
    random_state=42,
)

linear_regression.fit(X_reg_train, y_reg_train)
tree_regression.fit(X_reg_train, y_reg_train)
print("Regression models fitted.")


### Calculate regression metrics

All metrics use the same test rows.


In [ ]:
regression_rows = []
for name, model in {
    "Linear regression": linear_regression,
    "Decision tree regression": tree_regression,
}.items():
    predictions = model.predict(X_reg_test)
    regression_rows.append({
        "model": name,
        "MAE": mean_absolute_error(y_reg_test, predictions),
        "RMSE": mean_squared_error(y_reg_test, predictions) ** 0.5,
        "R2": r2_score(y_reg_test, predictions),
    })

regression_comparison = pd.DataFrame(regression_rows).set_index("model")
print(regression_comparison.round(2))


### Use cross-validation for regression

KFold with shuffling is suitable here because rows are independent synthetic stores, not time-ordered observations.


In [ ]:
regression_cv = KFold(n_splits=5, shuffle=True, random_state=42)
regression_scores = cross_validate(
    linear_regression,
    X_reg,
    y_reg,
    cv=regression_cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
)
print("CV MAE:", round(-regression_scores["test_mae"].mean(), 2))
print("CV RMSE:", round(-regression_scores["test_rmse"].mean(), 2))
print("CV R2:", round(regression_scores["test_r2"].mean(), 3))


## Guided selection checklist

Do not select solely by the largest score. Consider metric relevance, variability, calibration, subgroup errors, interpretability, data drift, latency, and the cost of mistakes. Refit the chosen approach on training data and evaluate once on the held-out test set.


### Create a compact evidence table

The table separates task type and evaluation design.


In [ ]:
evidence_table = pd.DataFrame([
    {
        "task": "Churn classification",
        "validation": "5-fold stratified CV",
        "primary_metric": "F1 and recall",
        "candidate": cv_comparison["f1_mean"].idxmax(),
    },
    {
        "task": "Weekly revenue regression",
        "validation": "Holdout + 5-fold CV",
        "primary_metric": "MAE",
        "candidate": regression_comparison["MAE"].idxmin(),
    },
])
print(evidence_table)


## Student coding lab

Use cross-validation for model comparison and reserve a separate test set for final confirmation.


        ### Student task 1: Build dense preprocessing

        <div class="task"><strong>Your job:</strong> Create the numeric and categorical preprocessing required by LDA, GaussianNB, and SVC.</div>

        **Check your work**

        - Numerical features are imputed and scaled.
- One-hot output is dense.
- Unknown categories are handled safely.


In [ ]:
# STUDENT TASK 1
dense_preprocessor = None
print(dense_preprocessor)


        ### Student task 2: Compare three classifiers

        <div class="task"><strong>Your job:</strong> Run five-fold stratified cross-validation for LDA, GaussianNB, and SVC. Report mean and standard deviation for F1 and recall.</div>

        **Check your work**

        - Preprocessing is fitted inside each fold.
- The same folds and metrics are used.
- Variation accompanies every mean.


In [ ]:
# STUDENT TASK 2
classifier_cv_results = []
print(pd.DataFrame(classifier_cv_results))


        ### Student task 3: Create regression data

        <div class="task"><strong>Your job:</strong> Generate at least 200 reproducible rows with four business predictors and one continuous target containing random noise.</div>

        **Check your work**

        - The seed is fixed.
- The target is continuous.
- The target uses predictors plus random noise.


In [ ]:
# STUDENT TASK 3
rng = np.random.default_rng(123)
task_regression_data = pd.DataFrame()
print(task_regression_data.head())


        ### Student task 4: Compare two regressors

        <div class="task"><strong>Your job:</strong> Split the regression data, fit linear and constrained tree models, and report MAE, RMSE, and R-squared.</div>

        **Check your work**

        - Both models use identical test rows.
- Metrics are calculated from predictions.
- MAE and RMSE retain the target unit.


In [ ]:
# STUDENT TASK 4
task_regression_results = pd.DataFrame()
print(task_regression_results)


        ### Student task 5: Validate regression with folds

        <div class="task"><strong>Your job:</strong> Run five-fold cross-validation for both regressors, convert negative error scorers back to positive errors, and recommend a candidate with uncertainty.</div>

        **Check your work**

        - Both models use the same KFold object.
- Mean and standard deviation are reported.
- The recommendation considers error, variation, and interpretability.



**Optional extension:** Create residual plots and investigate whether error size changes with predicted revenue.


In [ ]:
# STUDENT TASK 5
task_cv_results = pd.DataFrame()
print(task_cv_results)
# Recommendation:


        ## Knowledge check

        Answer these without running new code first.

        1. Why should preprocessing remain inside cross-validation?
2. How does stratified K-fold differ from ordinary K-fold?
3. When is regression appropriate?
4. How do MAE and RMSE respond differently to large errors?
5. Why preserve a final test set?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** TW15 removes labels and studies clustering, hierarchical methods, PCA, t-SNE, and validation of unsupervised results.
